In [ ]:
# A CNN for MNIST: recognizing handwritten digits (0-9).
#
# Each image is a 28x28 grayscale picture of ONE handwritten digit.
# The label is the digit itself: 0, 1, ..., 9  -> 10 classes.
# (The data was prepared in S03_mnist_data_prep.ipynb and saved to mnist_small.pickle.)
#
# What is new compared with the toy examples (S1_toy_cnn.py, S2_toy_mlp.py)?
#   - 10 classes instead of 2: the model outputs 10 logits (one per digit),
#     and we use CrossEntropyLoss instead of BCEWithLogitsLoss.
#   - Real data has to be scaled (pixel values 0-255 -> roughly mean 0, std 1).
#   - The data is bigger, so we train with mini-batches instead of all at once.
#   - We measure the F1-score (not only accuracy) with scikit-learn.

import pickle

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from torch import nn


In [ ]:
# (Google Colab only) Mount Google Drive so we can read files stored there.
# Skip this cell when running locally.
from google.colab import drive

drive.mount("/content/drive/")

In [ ]:
# ── 1. Data ────────────────────────────────────────────────────────────────
# Load the small MNIST dataset (8000 images) saved by S03_mnist_data_prep.ipynb.
# data is a dict: data["_X"] = images, data["_Y"] = labels.

filepath = "/content/drive/MyDrive/Colab Notebooks/mnist_small.pickle"
# filepath = "mnist_small.pickle"  # when running locally
with open(filepath, "rb") as file:
    data = pickle.load(file)

In [ ]:
# _X has shape (N, 1, 28, 28): (batch, channels, height, width).
#   1 channel because the images are grayscale (a color image would have 3).
# _Y has shape (N, 1): one digit label (0-9) per image.
# The leading "_" means "raw data, not scaled yet".
_X = data["_X"].astype(np.float64)
_Y = data["_Y"].astype(np.int32)
print(_X.shape)
print(_X.dtype)
print(_Y.shape)
print(_Y.dtype)

In [ ]:
# Visualize some samples
num_row = 10  # Number of for digits 0–9
num_col = 5  # Number of images per digit
fig, axes = plt.subplots(num_row, num_col, figsize=(6, 12))  # Create a grid of subplots

for digit in range(num_row):  # Loop over each digit (0–9)
    # Find all indices in _y_train where the label equals the current digit
    idx_arr = np.where(_Y == digit)[0]
    for col in range(num_col):  # For each column (number of samples shown)
        idx = idx_arr[col]  # Get the index for the corresponding sample
        # Extract and squeeze the image tensor (remove channel dimension if necessary)
        img = _X[idx, ...].squeeze(0)
        ax = axes[digit, col]  # Select the correct subplot
        ax.imshow(img, cmap="gray")  # Show the image in grayscale
        ax.set_title(f"Digit = {digit}")  # Set the plot title to the digit
        if col == 0:  # Only label the first column for clarity
            ax.set_ylabel(f"Digit {digit}", fontsize=12)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title("")
fig.tight_layout()  # Optimize spacing
plt.show()  # Display the plot

In [ ]:
# Split data into train and test sets
#   80% for training, 20% as fresh images the model never sees in training.
#   random_state=42 makes the split the same every time we run.
_X_train, _X_test, _Y_train, _Y_test = train_test_split(
    _X, _Y, test_size=0.2, random_state=42
)
print(_X_train.shape)  # (6400, 1, 28, 28)
print(_Y_train.shape)  # (6400, 1)
print(_X_test.shape)  # (1600, 1, 28, 28)
print(_Y_test.shape)  # (1600, 1)

In [ ]:
# A simple standard scaler: shift to mean 0 and scale to std 1.
# (Shown for reference -- below we do the same steps by hand so we can
#  reuse the TRAINING mean/std for the test set.)
def scaler_transform(X):
    _mean = np.mean(X)
    _std = np.std(X)
    return (X - _mean) / _std


In [ ]:
# Scale the data
# Note that we don't use SKLearn's StandardScaler here because our data is 4D and SKLearn's StandardScaler only works with 2D data.
# Therefore, we have to do it manually. We need to calculate mean and std of the training data.
# One mean and one std over ALL pixels of ALL training images (two single numbers).
# We compute them from the TRAINING data only, so no information leaks from the test set.
_mean = np.mean(_X_train)
_std = np.std(_X_train)
print(f"Mean: {_mean}, Std: {_std}")

In [ ]:
# Scale the X data
# The test set is scaled with the TRAINING mean/std (same rule for both sets).
X_train = (_X_train - _mean) / _std
X_test = (_X_test - _mean) / _std

# Check the range of X before and after scaling
#   before: pixel values 0 (black) to 255 (white)
#   after : roughly centered around 0 -- easier for the network to learn
print("_X_train (before scaling):", _X_train.min(), "to", _X_train.max())
print("X_train (after scaling):", X_train.min(), "to", X_train.max())

# No scaling for labels
# (they are class numbers 0-9, not measurements)
Y_train = _Y_train
Y_test = _Y_test


In [ ]:
# Convert to PyTorch tensors
# float32 is the default number type for PyTorch model weights.
X_train_pt = torch.tensor(X_train, dtype=torch.float32)
X_test_pt = torch.tensor(X_test, dtype=torch.float32)

# For labels, we need to convert them to long type because CrossEntropyLoss expects the target labels to be of type long.
# (long = integer class index, e.g. 7 means "digit 7")
Y_train_pt = torch.tensor(Y_train, dtype=torch.long)
Y_test_pt = torch.tensor(Y_test, dtype=torch.long)

In [ ]:
# ── 2. Model ───────────────────────────────────────────────────────────────
num_classes = 10
model = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),  # 1 input channel -> 16 filters
    nn.ReLU(),  # non-linearity
    nn.MaxPool2d(2),  # 28x28 -> 14x14: keep the strongest value in each 2x2 block
    nn.Conv2d(16, 32, kernel_size=3, padding=1),  # 16 channels -> 32 filters
    nn.ReLU(),  # non-linearity
    nn.MaxPool2d(2),  # 14x14 -> 7x7
    nn.AdaptiveAvgPool2d((4, 4)),  # 7x7 -> 4x4 (fixed output size)
    nn.Flatten(),  # (32, 4, 4) -> 512 numbers
    nn.Linear(32 * 4 * 4, num_classes),  # 512 features -> 10 logits
)

pytorch_total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters in the model: {pytorch_total_params}")

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
# (Optional) LeNet-5 -- uncomment this cell to use it instead of the model above.

# model = nn.Sequential(
#     nn.Conv2d(1, 6, kernel_size=5, padding=2),  # 1 input channel -> 6 filters
#     nn.Tanh(),  # non-linearity (the original LeNet used Tanh)
#     nn.AvgPool2d(2),  # 28x28 -> 14x14: average of each 2x2 block
#     nn.Conv2d(6, 16, kernel_size=5),  # no padding: 14x14 -> 10x10
#     nn.Tanh(),
#     nn.AvgPool2d(2),  # 10x10 -> 5x5
#     nn.Flatten(),  # (16, 5, 5) -> 400 numbers
#     nn.Linear(16 * 5 * 5, 120),  # 400 features -> 120
#     nn.Tanh(),
#     nn.Linear(120, 84),  # 120 features -> 84
#     nn.Tanh(),
#     nn.Linear(84, num_classes),  # 84 features -> 10 logits
# )
# pytorch_total_params = sum(p.numel() for p in model.parameters())
# print(f"Total number of parameters in the model: {pytorch_total_params}")

# optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Function to get batches of indices
# Mini-batch training: instead of feeding all 6400 images at once (as in the
# toy examples), we update the weights after every small batch of images.
# The indices are shuffled each epoch so the batches are different every time.
def get_batches_indices(num_samples, batch_size):
    indices = np.arange(num_samples)  # 0, 1, ..., num_samples - 1
    np.random.shuffle(indices)  # random order
    for start_idx in range(0, num_samples, batch_size):
        end_idx = min(start_idx + batch_size, num_samples)  # last batch may be smaller
        batch_indices = indices[start_idx:end_idx]
        yield batch_indices  # "yield" hands out one batch at a time


# Example usage
# 20 samples in batches of 6 -> 6, 6, 6, 2
num_samples = 20
batch_size = 6
for idxes in get_batches_indices(num_samples, batch_size):
    print(idxes)


In [ ]:
# Turn logits into predicted digits and compare them with the true labels.
#   logits -> softmax -> probabilities (10 per image, summing to 1)
#          -> argmax  -> predicted digit (the class with the highest probability)
# (This is the 10-class version of sigmoid + ">= 0.5" in the toy examples.)
def calc_metrices(logits: torch.Tensor, labels: torch.Tensor, isPrint=False):
    # Apply softmax to logits to obtain prediction probabilities for each class
    probs = F.softmax(logits, dim=1)  # Shape: (batch_size, num_classes)

    # Use argmax to get the predicted class index for each sample
    preds = torch.argmax(probs, dim=1)  # Shape: (batch_size)

    # Move prediction and label tensors to CPU and convert to numpy arrays for scikit-learn compatibility
    preds_np = preds.cpu().numpy()
    labels_np = labels.cpu().numpy()

    # Generate the classification report as a dictionary (includes precision, recall, f1, etc.)
    report = classification_report(labels_np, preds_np, output_dict=True)

    # Optionally print a nicely formatted classification report if isPrint is True
    if isPrint:
        print(classification_report(labels_np, preds_np, digits=3))

    # Return the report dictionary and the numpy arrays of predictions and labels
    return report, preds_np, labels_np

In [ ]:
# Lists to record the training loss and F1-score of every epoch (for plotting later)
loss_arr = []
F1_arr = []

In [ ]:
# ── 3. Train ───────────────────────────────────────────────────────────────
# One epoch = one pass over ALL training images, in mini-batches.
# 6400 images / batch size 10 = 640 weight updates per epoch.
n_epochs = 20  # number of epochs to run
batch_size = 10  # size of each batch

for epoch in range(n_epochs):
    # Training Phase
    model.train()  # put the model in training mode
    epoch_train_loss = 0.0
    epoch_train_f1 = 0.0
    logit_arr = []  # collect logits/labels of all batches to compute F1 for the epoch
    label_arr = []

    for idxes in get_batches_indices(len(X_train_pt), batch_size):
        X_batch_pt = X_train_pt[idxes]  # (batch, 1, 28, 28)
        Y_batch_pt = Y_train_pt[idxes]  # (batch, 1)
        optimizer.zero_grad()  # clear gradients from the previous step
        Y_pred = model(X_batch_pt)  # forward pass: (batch, 10) logits
        loss = loss_fn(
            Y_pred, Y_batch_pt.view(-1)
        )  # view(-1) is used to flatten the labels to the shape (batch_size, ) which is required by CrossEntropyLoss
        # Backward pass
        loss.backward()  # backpropagation: compute gradients

        # Optional gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping

        # Update weights
        optimizer.step()
        # Multiplies the average loss per sample by the number of
        # samples in the batch to get the total loss for this batch.
        epoch_train_loss += loss.item() * len(idxes)
        logit_arr.append(Y_pred)
        label_arr.append(Y_batch_pt)

    # Average loss per image over the whole epoch
    avg_train_loss = epoch_train_loss / len(X_train)
    loss_arr.append(avg_train_loss)

    # Join the batches back together and compute the F1-score for the epoch.
    # "weighted avg" = average F1 over the 10 digits, weighted by how many
    # images each digit has.
    logits = torch.concat(logit_arr, dim=0)
    labels = torch.concat(label_arr, dim=0)
    metrices, _, _ = calc_metrices(logits=logits, labels=labels.view(-1))
    avg_train_f1 = metrices["weighted avg"]["f1-score"]
    F1_arr.append(avg_train_f1)

    print(
        f"Epoch {epoch + 1}/{n_epochs}, Training Loss: {avg_train_loss:.4f}, Training F1-Score: {avg_train_f1:.4f}"
    )

In [ ]:
# Plot the learning curves: loss should go down, F1-score should go up.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: training loss per epoch
axes[0].plot(range(1, len(loss_arr) + 1), loss_arr, marker="o")
axes[0].set_title("Training Loss over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Average Training Loss")
axes[0].grid()

# Right: training F1-score per epoch
axes[1].plot(range(1, len(F1_arr) + 1), F1_arr, marker="o")
axes[1].set_title("Training F1-Score over Epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Average Training F1-Score")
axes[1].grid()

plt.tight_layout()
plt.show()

In [ ]:
# ── 4. Check ───────────────────────────────────────────────────────────────
# Evaluate on the TEST images to see if the model generalizes.
# Evaluation Phase
model.eval()  # put the model in evaluation mode
with torch.no_grad():  # no gradients needed: we are not training
    Y_pred_test = model(X_test_pt)  # (1600, 10) logits
    test_loss = loss_fn(Y_pred_test, Y_test_pt.view(-1)).item()
    report, preds_np, labels_np = calc_metrices(
        logits=Y_pred_test, labels=Y_test_pt.view(-1), isPrint=False
    )  # set isPrint=True to see precision/recall/F1 for each digit

print(f"Test Loss: {test_loss:.4f}")
print(f"Test F1-Score: {report['weighted avg']['f1-score']:.4f}")

In [ ]:
# ── 5. Peek at the mistakes ────────────────────────────────────────────────
# Investigate misclassified samples
# Find the test images where the predicted digit differs from the true digit.
misclassified_indices = np.where(preds_np != labels_np)[0]
print(f"Number of misclassified samples: {len(misclassified_indices)}")

# Visualize some misclassified samples
# Many of them are hard to read even for a human.
num_samples_to_show = min(10, len(misclassified_indices))
for i in range(num_samples_to_show):
    idx = misclassified_indices[i]
    plt.imshow(X_test[idx].squeeze(), cmap="gray")  # (1, 28, 28) -> (28, 28)
    plt.title(f"True Label: {labels_np[idx]}, Predicted Label: {preds_np[idx]}")
    plt.axis("off")
    plt.show()